# Testing the RAW attention-score capture on Qwen2.5-VL-7B (4-bit)

Same correctness suite as the PaliGemma / SmolVLM notebooks, run against
**`Qwen/Qwen2.5-VL-7B-Instruct`** loaded in **4-bit** (~5 GB) so it fits a single
16 GB GPU (T4 / g4dn).

The raw **pre-softmax** text→vision scores are captured the *generalisable* way
(identical to the SmolVLM path): we monkeypatch the module-level
`eager_attention_forward` in `transformers` to stash `Q Kᵀ * scaling (+mask)`
before softmax. The adapter fills the SAME `ProbeOutput`, so the SAME invariant
tests from `tests/test_raw_attention.py` run unchanged.

* **(A) Structure** — image/text token counts partition the sequence; the sliced
  text→vision map has exactly `n_text × n_image` elements.
* **(B) Raw-vs-softmax discriminators** — the captured tensor has negatives, its
  rows don't sum to 1, and it isn't the softmax map → it is raw logits.
* **(C) Ground-truth identity** — `softmax(raw)` reproduces the model's own
  post-softmax attention (`argmax` match + numeric reconstruction).

> **Runtime:** GPU runtime (`Runtime → Change runtime type → GPU`).
> **Scaling up:** set `QWEN_VL_ID=Qwen/Qwen2.5-VL-72B-Instruct` on a large
> multi-GPU node (72B needs ~40 GB even in 4-bit; it will NOT fit a single T4).

## 1. Install dependencies
`bitsandbytes` for 4-bit, `accelerate` for `device_map`.

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate bitsandbytes safetensors pillow num2words pytest

## 2. Clone the repo

In [ ]:
!rm -rf text_vision_attention_map          # force a fresh checkout (avoid a stale clone)
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 3. Build the Qwen2.5-VL probe
One forward pass captures BOTH the raw pre-softmax scores and the model's own
post-softmax attention, for the first few language-model decoder layers.
First run downloads + 4-bit-quantizes the 7B weights.

In [ ]:
import importlib.util, os

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

T = _load("test_raw_attention", "tests/test_raw_attention.py")
Q = _load("probe_qwen", "tests/probe_qwen.py")

o = Q.make_qwen_output()          # Qwen2.5-VL-7B-Instruct in 4-bit
assert o is not None, "Qwen probe failed to load — see the printed reason above (GPU / VRAM / version)."
print("Probe:", o.name)
print("Sequence length L      :", o.L)
print("Decoder layers captured:", sorted(o.raw_scores))
print("raw_scores[0] shape    :", tuple(o.raw_scores[sorted(o.raw_scores)[0]].shape), "(H, L, L)")

## 4. Run ALL test cases on the real Qwen2.5-VL probe
The exact same invariant tests used for PaliGemma / SmolVLM — unchanged.

In [ ]:
test_fns = [
    T.test_token_counts_partition_the_sequence,
    T.test_submatrix_element_count,
    T.test_raw_scores_contain_negative_values,
    T.test_raw_rows_do_not_sum_to_one,
    T.test_raw_differs_from_post_softmax,
    T.test_argmax_of_raw_matches_argmax_of_post,
    T.test_softmax_of_raw_reconstructs_post_softmax,
    T.test_within_row_log_difference_is_constant,
]

passed = 0
for fn in test_fns:
    try:
        fn(o)
        print(f"PASS  {fn.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")

print(f"\n{passed}/{len(test_fns)} tests passed on Qwen2.5-VL.")

## 5. (A) Structure — concrete numbers

In [ ]:
L0 = sorted(o.raw_scores)[0]
n_img = int(o.image_token_mask.sum())
n_txt = int(o.text_token_mask.sum())
print("image tokens :", n_img)
print("text  tokens :", n_txt)
print("image & text disjoint:", not bool((o.image_token_mask & o.text_token_mask).any()))
print("n_img + n_txt <= L    :", n_img + n_txt, "<=", o.L)

P = T.slice_text_vision(o.raw_scores[L0], o.text_token_mask, o.image_token_mask).mean(0)
print(f"\nlayer-{L0} text->vision submatrix shape:", tuple(P.shape), "(L_t, L_v)")
print("elements:", P.numel(), "==  n_txt * n_img =", n_txt * n_img)

## 6. (B) Raw-vs-softmax discriminators — concrete numbers

In [ ]:
import torch
L0 = sorted(o.raw_scores)[0]
raw0, post0 = o.raw_scores[L0], o.post_softmax[L0]
print("min raw value           :", round(raw0.min().item(), 4), " (< 0  => softmax impossible)")
print("raw row sums (first 5)  :", [round(v, 3) for v in raw0.sum(-1).flatten()[:5].tolist()],
      " (softmax rows would be 1.0)")
print("post-softmax row sum     :", round(post0.sum(-1).flatten()[0].item(), 4), " (~1.0)")
print("raw == post-softmax?     :", torch.allclose(raw0, post0, atol=1e-4), " (must be False)")

## 7. (C) Ground-truth identity — softmax(raw) == model's attention

In [ ]:
import torch
L0 = sorted(o.raw_scores)[0]
raw0 = o.raw_scores[L0].double()
post0 = o.post_softmax[L0].double()

allowed = post0 > 0
masked_raw = torch.where(allowed, raw0, torch.full_like(raw0, torch.finfo(torch.float64).min))
recon = torch.softmax(masked_raw, dim=-1)
max_diff = (recon - post0).abs()[allowed].max().item()
argmax_match = (raw0.argmax(-1) == post0.argmax(-1)).float().mean().item()

print("max |softmax(raw) - post_softmax| :", f"{max_diff:.2e}", " (~0  => exact reconstruction)")
print("argmax(raw) == argmax(post) frac  :", f"{argmax_match:.4f}", " (1.0 => same top key everywhere)")

## 8. What made this generalise?
Nothing in `tests/test_raw_attention.py` changed. Qwen2.5-VL reuses the generic
`tests/probe_hf_vlm.py` core (shared with SmolVLM) via a 6-line wrapper
`tests/probe_qwen.py`. To test yet another VLM, write one more wrapper that calls
`make_hf_vlm_output(model_id, ...)`; every test above runs on it.

**72B:** `import os; os.environ['QWEN_VL_ID']='Qwen/Qwen2.5-VL-72B-Instruct'`
before cell 3, on a node with ~40 GB+ of GPU memory.